# Structure Output, Tools & Agents

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-5-mini")
model.invoke("Hi")
print("Cinebot's brain is connected!")

Cinebot's brain is connected!


## Structured Output

In [4]:
booking_requests = [
    "Hi, I'd like 2 tickets for Interstellar at the 7pm show tonight, name is Priya.",
    "can u book me a seat for the 9:30 showing of dune part two? im rohan",
    "URGENT – need to CANCEL my booking for Oppenheimer, confirmation was under Aisha",
]

In [5]:
for msg in booking_requests:
    r = model.invoke(f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}")
    print(r.content)
    print("---")

{
  "name": "Priya",
  "movie": "Interstellar",
  "action": "book"
}
---
Name: Rohan
Movie: Dune Part Two
Action: Book (wants to book a seat for the 9:30 showing)
---
{
  "customer_name": "Aisha",
  "movie": "Oppenheimer",
  "request": "cancel"
}
---


## with_structured_output

In [6]:
from pydantic import BaseModel, Field
from typing import Literal

class BookingRequest(BaseModel):
    name: str = Field(description="The customer's name")
    movie: str = Field(description="The movie title")
    action: Literal["book", "cancel"] = Field(description="The action the customer wants to take")
    ticket_count: int = Field(description="How many tickets, default 1 if not mentioned", default=1)

print("Schema is defined")

Schema is defined


In [7]:
structured_model = model.with_structured_output(BookingRequest)

In [8]:
for msg in booking_requests:
    r = structured_model.invoke(f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}")
    print(r)
    print(f" --> action_type: {type(r.action)}, value: {r.action}")
    print("---")

name='Priya' movie='Interstellar' action='book' ticket_count=2
 --> action_type: <class 'str'>, value: book
---
name='Rohan' movie='Dune Part Two' action='book' ticket_count=1
 --> action_type: <class 'str'>, value: book
---
name='Aisha' movie='Oppenheimer' action='cancel' ticket_count=1
 --> action_type: <class 'str'>, value: cancel
---


In [9]:
print(type(r))
print(r)

<class '__main__.BookingRequest'>
name='Aisha' movie='Oppenheimer' action='cancel' ticket_count=1


## Tool Strategy & Provider Strategy

Two different mechanisms achieve the same guarantee. `ProviderStrategy` uses the model provider's own native structured-output feature (fast, but only works where supported). `ToolStrategy` fakes it via a synthetic tool call (works almost everywhere, slightly slower).

In [10]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

In [11]:
provider_strategy_model = model.with_structured_output(BookingRequest, provider_strategy=ProviderStrategy(BookingRequest))

In [12]:
from pydantic import BaseModel
from langchain.agents import create_agent


class Answer(BaseModel):
    summary: str
    confidence: float


agent = create_agent(model="openai:gpt-3.5-turbo", response_format=ToolStrategy(Answer))  # Will fail if ProviderStrategy is used
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
result["structured_response"]  # Answer(summary=..., confidence=...)

Answer(summary='Artificial Intelligence (AI) trends include the rise of AI-powered automation, increased adoption of machine learning and deep learning techniques, growth of AI in healthcare and finance, advancements in natural language processing and computer vision, ethical considerations surrounding AI development and deployment, integration of AI with Internet of Things (IoT) devices, use of AI for cybersecurity and fraud detection, emergence of AI in autonomous vehicles and robotics, and development of AI for personalized customer experiences and recommendation systems.', confidence=0.9)

In [13]:
from langchain.tools import tool

@tool
def peek_showtimes(movie_title: str) -> str:
    """Look up available showtimes for a movie."""
    # Some logic to get showtimes
    return "7:00 PM, 9:30 PM"

In [14]:
class NewBooking(BaseModel):
    """A request to book NEW tickets."""
    customer_name: str
    movie_title: str
    ticket_count: int


class CancelBooking(BaseModel):
    """A request to CANCEL an existing booking."""
    customer_name: str
    movie_title: str


from typing import Union

union_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[peek_showtimes],
    response_format=ToolStrategy(Union[NewBooking, CancelBooking])
)

In [15]:
result = union_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I want to cancel my movie Oppenheimer, I am Vivek"
        }
    ]
})

In [16]:
result['structured_response']

CancelBooking(customer_name='Vivek', movie_title='Oppenheimer')

In [17]:
result = union_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Book 2 tickets for Interstellar, name is Priya"
        }
    ]
})

In [18]:
result['structured_response']

NewBooking(customer_name='Priya', movie_title='Interstellar', ticket_count=2)

### Seat Booking agent

In [19]:
class SeatBooking(BaseModel):
    customer_name: str
    ticket_count: int = Field(
        description="Number of tickets, must be between 1 and 10",
        ge=1,
        le=10
    )

seat_agent = create_agent(
    model="openai:gpt-3.5-turbo",
    tools=[],
    response_format=ToolStrategy(SeatBooking),
    system_prompt="Extract the booking details exactly as stated, don't invent anything"
)

In [20]:

result = seat_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Hi I am Vivek, I want to give party to my students, book 15 tickets"
        }
    ]
})

In [21]:
result["structured_response"]

SeatBooking(customer_name='Vivek', ticket_count=10)

In [22]:
result

{'messages': [HumanMessage(content='Hi I am Vivek, I want to give party to my students, book 15 tickets', additional_kwargs={}, response_metadata={}, id='f001b7b7-b62f-46c1-8ad8-e233ee0d6c41'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 101, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EL4nNEkuDxt4cQbbyjz8ZZWUGyyhP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0765a-5f6d-7660-88a9-2119a7072776-0', tool_calls=[{'name': 'SeatBooking', 'args': {'customer_name': 'Vivek', 'ticket_count': 15}, 'id': 'call_xFcbG7jkjQTYVGJeXE7yfezI', 'type':

Initially it gave 15 tickets, then it validated as per the tool message then corrected the output

In [23]:
{
  "messages": [
    {
      "type": "HumanMessage",
      "content": "Hi I am Vivek, I want to give party to my students, book 15 tickets"
    },
    {
      "type": "AIMessage",
      "content": "",
      "finish_reason": "tool_calls",
      "tool_calls": [
        {
          "name": "SeatBooking",
          "args": {
            "customer_name": "Vivek",
            "ticket_count": 15
          }
        }
      ]
    },
    {
      "type": "ToolMessage",
      "name": "SeatBooking",
      "content": "Error: Failed to parse structured output for tool 'SeatBooking': ticket_count must be less than or equal to 10"
    },
    {
      "type": "AIMessage",
      "content": "",
      "finish_reason": "tool_calls",
      "tool_calls": [
        {
          "name": "SeatBooking",
          "args": {
            "customer_name": "Vivek",
            "ticket_count": 10
          }
        }
      ]
    },
    {
      "type": "ToolMessage",
      "name": "SeatBooking",
      "content": "Returning structured response: customer_name='Vivek' ticket_count=10"
    }
  ],
  "structured_response": {
    "customer_name": "Vivek",
    "ticket_count": 10
  }
}

{'messages': [{'type': 'HumanMessage',
   'content': 'Hi I am Vivek, I want to give party to my students, book 15 tickets'},
  {'type': 'AIMessage',
   'content': '',
   'finish_reason': 'tool_calls',
   'tool_calls': [{'name': 'SeatBooking',
     'args': {'customer_name': 'Vivek', 'ticket_count': 15}}]},
  {'type': 'ToolMessage',
   'name': 'SeatBooking',
   'content': "Error: Failed to parse structured output for tool 'SeatBooking': ticket_count must be less than or equal to 10"},
  {'type': 'AIMessage',
   'content': '',
   'finish_reason': 'tool_calls',
   'tool_calls': [{'name': 'SeatBooking',
     'args': {'customer_name': 'Vivek', 'ticket_count': 10}}]},
  {'type': 'ToolMessage',
   'name': 'SeatBooking',
   'content': "Returning structured response: customer_name='Vivek' ticket_count=10"}],
 'structured_response': {'customer_name': 'Vivek', 'ticket_count': 10}}

## Tools

In [24]:
response = model.invoke("Is Interstellar showing tonight at 7pm at the Downtown Cinema?")

In [25]:
response.content

'I don’t have live access to theater listings, so I can’t check real-time showtimes. Which “Downtown Cinema” do you mean (city or ZIP)? If you tell me the location I can:\n\n- give the theater’s phone number and website so you can confirm,\n- show likely places to check (Google Movies, Fandango, Atom Tickets), or\n- draft a quick message you can send to the theater asking whether Interstellar plays at 7:00 tonight.\n\nOr, if you prefer to check now, search Google for: Interstellar showtimes Downtown Cinema [city or ZIP], or go to Fandango/Atom Tickets and enter the theater name and your ZIP. Which option do you want?'

In [26]:
from pydantic import BaseModel

class MovieShowtime(BaseModel):
    movie_title: str
    showtime: str
    cinema_name: str
    
response = model.with_structured_output(MovieShowtime).invoke("Is Interstellar showing tonight at 7pm at the Downtown Cinema?")

In [27]:
response

MovieShowtime(movie_title='Interstellar', showtime="Unknown — I don't have access to live showtime data. Would you like me to look it up for the Downtown Cinema tonight at 7:00 PM?", cinema_name='Downtown Cinema')

In [28]:
from langchain_core.tools import tool

Tool --> Args with type hints

Tools are just glorified Functions/API Calls

In [29]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema.

    Args:
        movie_title: The exact title of the movie to check
    """
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }

    return fake_showtimes.get(
        movie_title.lower(),
        "No showtimes found for that title."
    )

In [30]:
@tool('book_seats', description='Book Cinema for a customer, use whenever customer wants to book/reserve a seat.')
def reserve(movie: str, seats: int) -> str:
    """Reserve Seats"""
    return f"Reserved {seats} seat for {movie}"

In [31]:
from langchain_tavily import TavilySearch

tools = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

### args_schema

In [32]:
from pydantic import Field
from typing import Literal

class SeatBookingInput(BaseModel):
    movie_title: str = Field(description="Exact Movie Title")
    seat_count: int = Field(description="Number of seats to book", ge=1, le=10)
    preferred_row: Literal['front', 'middle', 'back'] = Field(
        default='middle',
        description='Preferred seat row'
    )

In [33]:
@tool(args_schema=SeatBookingInput)
def book_seats(movie_title: str, seats: int, preferred_row: str, config: str) -> str:
    """Book Seats for a Movie"""
    return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

Don't use `config` & `runtime` , those are reserved keywords for the tool. 

It will give runtime error.

In [34]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )


@tool
def get_weather(
    location: str,
    # config: str = "default",
    # runtime: str = "now",
    units: str = "celsius",
    include_forecast: bool = False   
) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    # result = f"Current weather in {location}: {temp} degrees {config} {runtime} {units[0].upper()}"
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"

    if include_forecast:
        result += "\nNext 5 days: Sunny"

    return result

In [35]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather]
)

In [36]:
results = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Delhi in celcius and tell me the forecast?"}]})

In [37]:
results

{'messages': [HumanMessage(content='What is the weather in Delhi in celcius and tell me the forecast?', additional_kwargs={}, response_metadata={}, id='8089d608-2a73-48d9-925a-ecb4c0e8f722'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 163, 'total_tokens': 327, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EL4nc4hQPod9j7ZbQHs62roSbsZk8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0765a-9d2e-7462-9994-6d72b3ff38f1-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Delhi, India', 'units': 'celsius', 'include_forecast': True}, 'id': 'call_r2

## Binding vs Executions

In [38]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    return "Show is available"

The below code won't call tool, agent will call tool.

In [39]:
model_with_tools = model.bind_tools([check_showtimes, book_seats])

In [40]:
responsee_with_bind_tools = model_with_tools.invoke("Check showtimes for Interstellar")

In [41]:
responsee_with_bind_tools

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 211, 'total_tokens': 237, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EL4nfvLMxaiQCLTDOIMjGyosmXNLl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0765a-a9c5-71c1-bad2-fa1ef87dbb39-0', tool_calls=[{'name': 'check_showtimes', 'args': {'movie_title': 'Interstellar'}, 'id': 'call_Zs8r5sZja1y7g3g341OUMwwe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 211, 'output_tokens': 26, 'total_tokens': 237, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio'

## Runtime in Tools

a runtime param in tool, which our tool can use to read a lots of things in the code and otherwise as well.

In [42]:
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage


@tool
def get_last_movie_mentioned(movie: str, runtime: ToolRuntime) -> str:
    """Get the last movie mentioned in the conversation."""
    pass

print(get_last_movie_mentioned.args)

{'movie': {'title': 'Movie', 'type': 'string'}}


In [43]:
from typing import Any
from langgraph.store.memory import InMemoryStore


loyalty_store = InMemoryStore()


@tool
def save_favourite_genres(
    customer_id: str,
    genre: str,
    runtime: ToolRuntime
) -> str:
    """Save a customer's favourite movie genre for future visits."""

    runtime.store.put(
        (customer_id, "preferences"),
        "favourite_genre",
        {"value": genre}
    )

    return f"Got it — I will remember you like {genre} movies"

@tool
def recall_favourite_genre(customer_id: str, runtime: ToolRuntime) -> str:
    """Recall a customer's fav movie genre, if we have saved it before"""
    favourite_genre = runtime.store.get(
        (customer_id, "preferences"),
        "favourite_genre"
    )
    return (
        favourite_genre.value["value"]
        if favourite_genre
        else "We don't have any saved preference for this user"
    )


memory_agent = create_agent(
    model=model,
    tools=[save_favourite_genres, recall_favourite_genre],
    store=loyalty_store # Attached to the agent, tools can access it using runtime
)

In [44]:
result = memory_agent.invoke({"messages": [("user", "Hi I'm customer priya_01, I love sci-fi movies, please remember that")]})

In [45]:
result = memory_agent.invoke({"messages": [("user", "What is my favourite movie genre? I'm priya_01")]})

In [46]:
print(result["messages"][-1].content)

I have "sci-fi" saved as your favourite movie genre. Would you like recommendations or to update it to something else?


## Skipping the Model's Final Polishing

In [47]:
@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Tickets are refundable up to 2 hours before showtime. No refunds after that."


direct_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_exact_refund_policy]
)

result = direct_agent.invoke({
    "messages": [
        ("user", "What's your refund policy? Please explain in points")
    ]
})

print(result["messages"][-1].content)

Tickets are refundable up to 2 hours before showtime. No refunds after that.


## Multiple Structured Output

In [48]:
from pydantic import BaseModel, Field
from typing import Union, Literal
from langchain.agents.structured_output import ToolStrategy


class NewTripRequest(BaseModel):
    """A request to plan a new trip."""
    user_id: str
    destination: str
    start_date: str = Field(description="Format: YYYY-MM-DD")
    end_date: str = Field(description="Format: YYYY-MM-DD")


class ModifyTripRequest(BaseModel):
    """A request to change an existing trip."""
    user_id: str
    trip_id: int
    change_description: str


class CancelTripRequest(BaseModel):
    """A request to cancel an existing trip."""
    user_id: str
    trip_id: int

In [ ]:
tripmate_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[
        get_real_weather,
        tavily_search,
        save_trip,
        get_saved_trips
    ],
    system_prompt="You are TripMate, a friendly trip booking assistant.",
    checkpointer=checkpointer,
    response_format=ToolStrategy(
        Union[
            NewTripRequest,
            ModifyTripRequest,
            CancelTripRequest
        ]
    )
)

<!-- tripmate_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[
        get_real_weather,
        tavily_search,
        save_trip,
        get_saved_trips
    ],
    system_prompt="You are TripMate, a friendly trip booking assistant.",
    checkpointer=checkpointer
) -->